In [1]:
import os, sys, warnings, pandas as pd, psycopg
from dotenv import load_dotenv
sys.path.insert(0, "..")
from collector.embed import clean_for_embedding        # strips Steam's [b], [url=...] tags for easier reading
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)
def q(sql, **params): return pd.read_sql(sql, conn, params=params)

LABELS = "../data/labels"
TEST_LABEL = f"{LABELS}/patch_test_to_label.csv"
TEST_KEY = f"{LABELS}/patch_test_key.csv"

In [2]:
CUTOFF = "2026-10-23"    # fixed, so a rerun draws the same posts even as new ones arrive
QUOTA = {"none": 30, "keyword_no_tag": 10, "tag": 5, "version_only": 5}
dev = pd.read_csv(f"{LABELS}/patch_sample_key.csv", encoding="utf-8-sig", dtype={"gid": str})

pool = q("""
WITH s AS (
    SELECT f.gid, f.appid, g.name, f.post_date, f.title, n.url, n.contents,
           CASE WHEN f.rule_tag     THEN 'tag'
                WHEN f.rule_keyword THEN 'keyword_no_tag'
                WHEN f.rule_version THEN 'version_only'
                ELSE 'none' END AS stratum
    FROM news_patch_flags f JOIN news n USING (gid) JOIN games g ON g.appid = f.appid
    WHERE f.post_date < %(cutoff)s::date AND NOT (f.gid = ANY(%(dev)s))
), one_per_game AS (
    SELECT *, row_number() OVER (PARTITION BY stratum, appid ORDER BY md5(gid)) AS nth FROM s
)
SELECT *, row_number() OVER (PARTITION BY stratum ORDER BY md5(gid)) AS k
FROM one_per_game WHERE nth = 1
""", cutoff=CUTOFF, dev=dev.gid.tolist())

test = pd.concat([pool[(pool.stratum == s) & (pool.k <= n)] for s, n in QUOTA.items()])
assert len(test) == sum(QUOTA.values()), test.stratum.value_counts()

if os.path.exists(TEST_LABEL) or os.path.exists(TEST_KEY):
    print("test files already exist; not overwriting")
else:
    test = test.sample(frac=1, random_state=20261023)                      # shuffle: no blocks by group
    test["excerpt"] = test.contents.fillna("").map(clean_for_embedding).str[:1500]
    test.sort_values("gid")[["gid", "stratum"]].to_csv(TEST_KEY, index=False, encoding="utf-8-sig")
    (test[["gid", "name", "post_date", "title", "url", "excerpt"]]
         .assign(is_patch_true="", note="")
         .to_csv(TEST_LABEL, index=False, encoding="utf-8-sig"))
    print(f"wrote {len(test)} posts to label")

wrote 50 posts to label


In [11]:
lab = pd.read_csv(TEST_LABEL, encoding="utf-8-sig",
                  dtype={"gid": str, "note": str, "is_patch_true": "Int64"})
START = 0                                            # 0, 10, 20, 30, 40
for i, r in lab.iloc[START:START + 10].iterrows():
    print(f"[{i}] {r['name']} | {r.post_date} | {r.title}\n     {r.url}\n{r.excerpt}\n{'-' * 100}")

[0] Kingdom Rush 6: Genesis TD | 2026-02-26 | Ask the Kingdom Rush 6 Devs Anything!
     https://steamstore-a.akamaihd.net/news/externalpost/steam_community_announcements/1825727806707604
[p]So… it’s real. Kingdom Rush 6: Genesis is happening.
[/p][p]Making a new Kingdom Rush is wild in the best way. This series has grown so much thanks to you, and we’re ridiculously grateful for that. To celebrate, we’re doing a Dev AMA.[/p][p] On March 3 at 12 PM (GMT-3) , the team will jump into this Steam thread and answer your questions. Development stories? Early lore hints? Design choices? Cut ideas? Random nonsense? Fair game.[/p][p] Drop your questions below and react to your favorites. We’ll answer as many as we can.[/p][p][/p][p]And if you’re ready to rewrite a bit of Linirea’s history with us, wishlist Kingdom Rush 6: Genesis and follow the journey. It really helps more than you think![/p][p][/p][p][dynamiclink href="https://store.steampowered.com/app/4259190/Kingdom_Rush_6_Genesis_TD"][/dy

In [12]:
answers = {
    0: (0, "game and chat announcement"),
    1: (0, "game announcement"),
    2: (1, "bug fixes and balance changes"),
    3: (0, "recap; change announced earlier in its own post"),
    4: (0, "game announcement"),
    5: (1, "added content"),
    6: (1, "new content and bug fixes"),
    7: (0, "sale announcement"),
    8: (0, "game announcement"),
    9: (0, "developer livestream announcement"),
    10: (1, "content, bug fixes, and balance changes"),
    11: (0, "participation announcement"),
    12: (1, "content, bug fixes, and balance changes"),
    13: (0, "test build announcement"),
    14: (0, "sale announcement"),
    15: (1, "content and event announcement"),
    16: (1, "bug fixes and quality of life changes"),
    17: (0, "test server announcement"),
    18: (0, "bundle announcement"),
    19: (1, "bug fixes"),
    20: (1, "content, quality of life, bug fixes, balance changes, DLC announcement"),
    21: (0, "future patch and game announcement"),
    22: (1, "bug fixes and quality of life changes"),
    23: (1, "bug fixes"),
    24: (0, "competition announcement, no game changes"),
    25: (1, "bug fixes"),
    26: (1, "content and DLC announcement"),
    27: (0, "merch announcement, no game changes"),
    28: (1, "content, new season"),
    29: (1, "hotfix"),
    30: (1, "content update"),
    31: (0, "trailer announcement"),
    32: (0, "future maintenance period announcement"),
    33: (0, "update trailer announcement"),
    34: (0, "thank you announcement"),
    35: (0, "thank you announcement"),
    36: (1, "content update"),
    37: (1, "content, bug fixes"),
    38: (0, "demo release announcement"),
    39: (1, "hotfix"),
    40: (1, "content and bug fixes"),
    41: (1, "flash update, scheduled same day, changes listed; content"),
    42: (0, "lore announcement"),
    43: (0, "merch announcement"),
    44: (0, "free DLC update announcement"),
    45: (1, "bug fixes and improvements"),
    46: (0, "art announcement"),
    47: (0, "merch announcement"),
    48: (0, "free weekend announcement"),
    49: (1, "content and event announcement")
}

done = sorted(answers)
assert all(v[0] in (0, 1) for v in answers.values())
lab.loc[done, "is_patch_true"] = [answers[i][0] for i in done]
lab.loc[done, "note"] = [answers[i][1] for i in done]
lab[["gid", "name", "post_date", "title", "url", "excerpt", "is_patch_true", "note"]].to_csv(
    TEST_LABEL, index=False, encoding="utf-8-sig")
print(f"{len(done)} of {len(lab)} labeled")

50 of 50 labeled
